# Neurovox Colab Runner

Thin, idempotent runner notebook for executing Neurovox training and benchmark workloads on Google Colab (Tesla T4 GPU).

### Core Principles:
1. **Idempotency:** Safe to run or re-run after kernel disconnects or preemptions.
2. **Single-file archive transfer:** Fast dataset unpacking from Drive to local fast SSD (`/content`), avoiding slow per-file Drive I/O.
3. **Drive Checkpoint Sync:** Checkpoints and logs are mirrored back to Google Drive every epoch.
4. **No Secrets:** Never put API tokens, private keys, or credentials in this notebook.
5. **Compute Units Tracking:** Monitor active compute unit burn rate via Colab's Resource panel.

In [ ]:
# Step 1: Verify assigned GPU
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive idempotently
import os
from pathlib import Path

try:
    from google.colab import drive
    drive_mount_path = Path('/content/drive')
    if not (drive_mount_path / 'MyDrive').exists():
        drive.mount(str(drive_mount_path))
    print("Google Drive is mounted.")
except ImportError:
    print("Not in a Google Colab environment.")

In [ ]:
# Step 3: Clone or update Neurovox repository
repo_path = Path("/content/Neurovox")
if not repo_path.exists():
    !git clone https://github.com/aanyavarshney/Neurovox.git {repo_path}
else:
    print("Repository exists. Pulling latest changes...")
    %cd {repo_path}
    !git pull

%cd {repo_path}

In [ ]:
# Step 4: Install dependencies in editable mode
!pip install --upgrade pip
!pip install -e ".[dev]"

In [ ]:
# Step 5: Configure storage paths via environment variables
# Setting these variables prevents any hardcoded paths in the codebase.
os.environ["DATA_ROOT"] = "/content/data"
os.environ["CACHE_ROOT"] = "/content/cache"
os.environ["CKPT_ROOT"] = "/content/checkpoints"
drive_sync_dir = Path("/content/drive/MyDrive/Neurovox_Sync")
drive_sync_dir.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT:        {os.environ['DATA_ROOT']}")
print(f"CACHE_ROOT:       {os.environ['CACHE_ROOT']}")
print(f"CKPT_ROOT:        {os.environ['CKPT_ROOT']}")
print(f"DRIVE_SYNC_DIR:   {drive_sync_dir}")

In [ ]:
# Step 6: Single-file transfer of dataset archive from Drive to local /content (fast NVMe)
local_data = Path(os.environ["DATA_ROOT"])
local_data.mkdir(parents=True, exist_ok=True)

drive_archive = drive_sync_dir / "dev-clean.tar.gz"
if drive_archive.exists() and not (local_data / "LibriSpeech").exists():
    print(f"Extracting {drive_archive} into {local_data}...")
    !tar -xzf {drive_archive} -C {local_data}
    print("Extraction complete.")
elif not (local_data / "LibriSpeech").exists():
    print("Archive not found in Drive. Downloading dev-clean directly to /content...")
    !curl -L https://www.openslr.org/resources/12/dev-clean.tar.gz -o /content/dev-clean.tar.gz
    !tar -xzf /content/dev-clean.tar.gz -C {local_data}
    !rm /content/dev-clean.tar.gz
    print("Downloaded and extracted.")
else:
    print("Dataset already exists at local storage location.")

In [ ]:
# Step 7: Run environment and CUDA verification
!python scripts/check_env.py

In [ ]:
# Step 8: Run hardware throughput benchmark
# Measures fp32 vs fp16 (with GradScaler) vs bf16 on Tesla T4
!python scripts/benchmark_hardware.py --duration-per-test 12.0 --output-json docs/benchmarks_colab_t4.json

In [ ]:
# Step 9: Example training invocation with --resume support
# Paste command to run, e.g.:
# !python scripts/train.py --config configs/t1_skahnn_s0_iaam_libri_clean_s42.yaml --resume

# Synchronize checkpoints to Google Drive:
local_ckpt = Path(os.environ["CKPT_ROOT"])
if local_ckpt.exists():
    drive_ckpt_dest = drive_sync_dir / "checkpoints"
    drive_ckpt_dest.mkdir(parents=True, exist_ok=True)
    print(f"Syncing checkpoints to {drive_ckpt_dest}...")
    !cp -ru {local_ckpt}/* {drive_ckpt_dest}/
    print("Checkpoints successfully synced.")